# SE-HOPE native-grid solar filament baseline

This notebook reproduces a frozen S1 baseline for Solar Filament Segmentation 2026. It uses only the supplied H-alpha pixels for inference. It does not upload predictions or start another job. The decoder was trained on eight images; this is a first-entry baseline with known false positives and fragmentation.

## Data and split

Attach the competition dataset and select a T4 GPU. The immutable plan verifies every input JPEG digest, the annotation digest and the original-image split. Fold 0 has 400 available training originals and 201 validation originals; 106 originals are held out. Repeated annotator records remain in the same original-image partition. The selected 1,024-update checkpoint used eight training originals and two validation originals for initial screening. Holdout predictions are produced only after the development selection lock; their scores do not tune this entry.

## Model and training

Native 2048×2048 image → 25 overlapping 512×512 tiles (stride 384) → frozen VisionHOPE-Tiny stride-4/8/16/32 features → width-128 FPN → three-layer, 64-query decoder → pixel masks → overlap association and tapered fusion → full-image COCO RLE.

The source bundle contains the complete training implementation and actual configuration. Training uses seed 42, AdamW at 3e-4, four accumulated patches, uniform original/annotator sampling, 50% positive and 50% uniform patches, and Hungarian matching with object BCE, balanced mask BCE and Dice loss. Positive target cropping is training-only; the inference grid never sees labels. The published decoder checkpoint permits prediction reproduction without repeating model selection or training.

## Fixed environment and inference

The following cell unpacks a SHA256-checked source archive and creates an isolated Python 3.11 / PyTorch 2.6.0+cu124 environment, preserving Kaggle system packages. The public pinned backbone and selected decoder are hash checked. The official CUDA backend is required; CPU/GPU equivalence is not claimed. The initial CUDA compilation can take several minutes. The entire job is bounded to one hour. The archive is embedded to preserve exact executable sources; readable modules are included in the public Git repository. No images, annotation masks or credentials are embedded.

In [ ]:
"""Private free-GPU pilot bootstrap. Filled with an allowlisted source bundle."""
import base64
import hashlib
import io
import importlib.metadata
import importlib.util
import json
import os
import signal
import re
import subprocess
import sys
import threading
import time
import urllib.request
import zipfile
from pathlib import Path, PurePosixPath

BUNDLE_B64 = '...'
BUNDLE_SHA256 = '8c5bd714d294810f74d3de17fc0d38ec6191f9c2d55e82685044530fd0dba944'
SUPPORTED = {'numpy': ('numpy', (1,26,0), (3,0,0)),
             'Pillow': ('PIL', (11,0,0), (13,0,0)),
             'scipy': ('scipy', (1,14,0), (2,0,0)),
             'pycocotools': ('pycocotools', (2,0,10), (3,0,0)),
             'psutil': ('psutil', (5,9,0), (8,0,0))}


def dependency_plan(installed):
    """Keep compatible platform binaries; never try a local Python3.11 pin on3.13."""
    needed = []
    for name, (_, lower, upper) in SUPPORTED.items():
        version = tuple(map(int, re.findall(r'\d+', installed.get(name, '0'))[:3]))
        version += (0,) * (3 - len(version))
        if not lower <= version < upper:
            needed.append(name + '>=' + '.'.join(map(str, lower)) + ',<' + '.'.join(map(str, upper)))
    return needed


def unpack_bundle(payload, expected_sha, destination):
    if hashlib.sha256(payload).hexdigest() != expected_sha:
        raise ValueError('Bundle digest differs.')
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        entries = archive.infolist()
        if len({entry.filename for entry in entries}) != len(entries):
            raise ValueError('Duplicate bundle paths.')
        if sum(entry.file_size for entry in entries) > 5 * 2**20:
            raise ValueError('Source bundle exceeds its size bound.')
        for entry in entries:
            p = PurePosixPath(entry.filename)
            if p.is_absolute() or '..' in p.parts or '\\' in entry.filename:
                raise ValueError('Nonrelative bundle path.')
            if (entry.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('Bundle symlinks are not allowed.')
        destination.mkdir(parents=True, exist_ok=False)
        archive.extractall(destination)
    manifest = json.loads((destination / 'bundle_manifest.json').read_text())
    actual = {str(p.relative_to(destination)): hashlib.sha256(p.read_bytes()).hexdigest()
              for p in destination.rglob('*') if p.is_file() and p.name != 'bundle_manifest.json'}
    if actual != manifest['files']:
        raise ValueError('Extracted source differs from the allowlisted manifest.')
    return manifest


import shutil
import tempfile

def main():
    root=Path('/kaggle/working');start=time.monotonic();child=None;scratch=None
    report={'status':'starting','optimizer_updates':0,'paid_resources_requested':False,
            'platform_timeout_seconds':3600,'bundle_sha256':BUNDLE_SHA256}
    def save():
        report['wall_seconds']=time.monotonic()-start
        (root/'inference_bootstrap_report.json').write_text(json.dumps(report,indent=2)+'\n')
    def alarm(*_):raise TimeoutError('Backend bootstrap exceeded 3400 seconds.')
    def run(command,limit,cwd=None,env=None):
        nonlocal child
        with (root/'inference_setup_and_run.log').open('a') as log:
            child=subprocess.Popen(command,cwd=cwd,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,start_new_session=True)
            def tee():
                for line in child.stdout:
                    log.write(line);log.flush();print(line,end='',flush=True)
            reader=threading.Thread(target=tee,daemon=True);reader.start()
            code=child.wait(timeout=min(limit,3380-(time.monotonic()-start)));reader.join(timeout=5)
            if code:raise subprocess.CalledProcessError(code,command)
    signal.signal(signal.SIGALRM,alarm);signal.setitimer(signal.ITIMER_REAL,3400)
    try:
        if not root.is_dir() or not Path('/kaggle/input').is_dir():raise RuntimeError('Kaggle allocation required.')
        source=root/'backend_source';unpack_bundle(base64.b64decode(BUNDLE_B64,validate=True),BUNDLE_SHA256,source)
        probe=subprocess.run([sys.executable,'-c','import torch,json,psutil,shutil; assert torch.cuda.is_available(); print(json.dumps(dict(gpu=torch.cuda.get_device_name(0),capability=list(torch.cuda.get_device_capability(0)),free_gpu_bytes=torch.cuda.mem_get_info(0)[0],host_available_bytes=psutil.virtual_memory().available,free_scratch_bytes=shutil.disk_usage("/tmp").free)))'],check=True,capture_output=True,text=True,timeout=45)
        report['resource_precheck']=json.loads(probe.stdout);save()
        pre=report['resource_precheck']
        if pre['capability']!=[7,5] or pre['free_gpu_bytes']<5*2**30 or pre['host_available_bytes']<6*2**30 or pre['free_scratch_bytes']<16*2**30:
            raise RuntimeError('Expected T4 / 5GiB CUDA / 6GiB host / 16GiB scratch preconditions.')
        scratch=Path(tempfile.mkdtemp(prefix='filament-backend-'))
        env=dict(os.environ,UV_CACHE_DIR=str(scratch/'uv-cache'),UV_PYTHON_INSTALL_DIR=str(scratch/'python-runtimes'),
                 TORCH_EXTENSIONS_DIR=str(scratch/'torch-extensions'),TORCH_CUDA_ARCH_LIST='7.5',MAX_JOBS='2',
                 CUDA_VISIBLE_DEVICES='0',OMP_NUM_THREADS='2',MKL_NUM_THREADS='2',PYTHONUNBUFFERED='1')
        report['status']='isolated_environment_setup';save();print(json.dumps(report),flush=True)
        run([sys.executable,'-m','pip','install','--no-cache-dir','--target',str(scratch/'tools'),'uv==0.12.19'],120,env=env)
        uv=str(scratch/'tools/bin/uv');python=str(scratch/'venv/bin/python')
        run([uv,'venv','--python','3.11',str(scratch/'venv')],120,env=env)
        run([uv,'pip','install','--python',python,'torch==2.6.0','torchvision==0.21.0','--index-url','https://download.pytorch.org/whl/cu124'],900,env=env)
        run([uv,'pip','install','--python',python,'-r',str(source/'requirements-cuda-benchmark.txt')],180,env=env)
        run([uv,'pip','check','--python',python],30,env=env)
        env['PYTHONPATH']=str(source/'src')+os.pathsep+str(source/'external')
        info=json.loads((source/'vendor/sehope_snapshot/weights_manifest.json').read_text());weights=scratch/'weights.pth'
        with urllib.request.urlopen(info['url'],timeout=45) as response,weights.open('wb') as target:
            while chunk:=response.read(2**20):target.write(chunk)
        if weights.stat().st_size!=info['size_bytes'] or hashlib.sha256(weights.read_bytes()).hexdigest()!=info['sha256']:
            raise ValueError('Pinned weight digest differs.')
        plan=json.loads((source/'submission_stage_plan.json').read_text())
        matches=[p for p in Path('/kaggle/input').rglob('checkpoint.pt') if hashlib.sha256(p.read_bytes()).hexdigest()==plan['checkpoint_sha256']]
        if plan.get('public_checkpoint_url'):
            checkpoint=scratch/'decoder.pt'
            with urllib.request.urlopen(plan['public_checkpoint_url'],timeout=45) as response,checkpoint.open('wb') as target:
                while chunk:=response.read(2**20):target.write(chunk)
            if hashlib.sha256(checkpoint.read_bytes()).hexdigest()!=plan['checkpoint_sha256']:raise ValueError('Public checkpoint digest differs.')
            matches=[checkpoint]
        if len(matches)!=1:raise ValueError('Expected exactly one attached checkpoint with the selected digest.')
        train=[p for p in Path('/kaggle/input').rglob('train_images') if p.is_dir()]
        test=[p for p in Path('/kaggle/input').rglob('test_images') if p.is_dir()]
        annotations=list(Path('/kaggle/input').rglob('MAGFiLO_1.0_Annotations_kaggle2026_train.json'))
        if len(train)!=1 or len(test)!=1 or len(annotations)!=1:raise ValueError('Ambiguous competition input.')
        report['status']='inference_started';save();print(json.dumps(report),flush=True)
        available=int(3340-(time.monotonic()-start))
        if available<3000:raise TimeoutError('Insufficient inference window.')
        run([python,str(source/'scripts/run_submission_stage.py'),'--plan',str(source/'submission_stage_plan.json'),
             '--checkpoint',str(matches[0]),'--source-root',str(source/'vendor/sehope_snapshot'),
             '--official-root',str(source/'vendor/visionhope_cuda/visionhope'),'--weights',str(weights),
             '--train-images',str(train[0]),'--test-images',str(test[0]),'--annotations',str(annotations[0]),
             '--split',str(source/'manifests/splits_final_v02.json'),'--output',str(root/'inference_run'),
             '--max-seconds',str(available-15)],available,cwd=source,env=env)
        result=json.loads((root/'inference_run/report.json').read_text())
        if result['status']!='completed' or not result['fresh_session_replay_exact']:raise RuntimeError('Incomplete inference/replay.')
        report.update(status='completed',stage_report_sha256=hashlib.sha256((root/'inference_run/report.json').read_bytes()).hexdigest(),
                      phase=plan['phase'],competition_submission_performed=False)
    except Exception as e:
        report.update(status='failed',error_type=type(e).__name__,error=str(e));raise
    finally:
        if child is not None and child.poll() is None:
            os.killpg(child.pid,signal.SIGTERM)
            try:child.wait(timeout=5)
            except subprocess.TimeoutExpired:os.killpg(child.pid,signal.SIGKILL);child.wait(timeout=5)
        signal.setitimer(signal.ITIMER_REAL,0);save()
        if scratch is not None:shutil.rmtree(scratch)
        print(json.dumps(report),flush=True)

if __name__=='__main__':main()


## Verify outputs

A fresh-session replay must reproduce the two screening-image predictions byte for byte. Every native image must have full pixel coverage, valid nonempty instance masks and unique instance IDs. Images with no detections have zero CSV rows and an explicit manifest entry. Evaluation runs only after the corresponding predictions are complete, and reports both public-arithmetic and strict one-to-one PQ.

In [ ]:
import json
from pathlib import Path
result = json.loads(Path('/kaggle/working/inference_run/report.json').read_text())
assert result['status'] == 'completed'
assert result['fresh_session_replay_exact']
print(json.dumps(result, indent=2))


## Interpretation

The printed scores describe their explicitly named splits. A Kaggle leaderboard score is separate and cannot be inferred from this notebook. Lower-level shape refiners, topology losses and broader training remain unimplemented improvements. The official final report and organizer form are separate submission requirements.